In [2]:
# ============================================================
# MULTI-STOCK MEAN-REVERSION TESTER
# Same strategy we optimized on NOW
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
START_DATE      = "2025-01-01"
INITIAL_CAPITAL = 1000.0

Z_ENTRY         = -1.5
Z_EXIT          = 0.0
ATR_MULT        = 2.0
SMA_WINDOW      = 20
ATR_WINDOW      = 14

TICKERS = [
    # Tier 1 - Software / Cloud (most similar to NOW)
    "NOW", "CRM", "ADBE", "SNOW", "DDOG", "CRWD", "NET", "PANW",
    # Tier 2 - Big Tech
    "GOOGL", "META", "AMZN", "MSFT", "NVDA", "AAPL", "TSLA",
    # Extra
    "NFLX", "ISRG"
]
# ======================================================


def get_data(ticker, start):
    try:
        df = yf.download(ticker, start=start, multi_level_index=False,
                         progress=False, auto_adjust=True)
        if df.empty or len(df) < 100:
            return None
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
        df.index = pd.to_datetime(df.index)
        return df
    except:
        return None


def add_indicators(df):
    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()
    return df


def run_strategy(df, initial_capital=INITIAL_CAPITAL):
    df = add_indicators(df)

    cash = float(initial_capital)
    shares = 0.0
    entry_price = 0.0
    stop_price = 0.0
    entry_date = None
    trades = []
    equity_curve = []

    start_i = max(SMA_WINDOW, ATR_WINDOW) + 1

    for i in range(start_i, len(df)):
        date = df.index[i]
        close = float(df['Close'].iloc[i])
        z = float(df['zscore'].iloc[i]) if pd.notna(df['zscore'].iloc[i]) else 0.0
        atr = float(df['atr'].iloc[i]) if pd.notna(df['atr'].iloc[i]) else 0.0

        # Entry
        if shares == 0 and z < Z_ENTRY and atr > 0 and cash > 10:
            shares = cash / close
            entry_price = close
            stop_price = close - (ATR_MULT * atr)
            entry_date = date
            cash = 0.0

        # Manage
        elif shares > 0:
            new_stop = close - (ATR_MULT * atr)  # Close-based (high-performing version)
            if new_stop > stop_price:
                stop_price = new_stop

            reason = None
            if close <= stop_price:
                reason = "ATR Trail"
            elif z >= Z_EXIT:
                reason = "Z > 0"

            if reason:
                proceeds = shares * close
                ret_pct = (close / entry_price - 1.0) * 100.0
                trades.append({
                    'return_pct': ret_pct,
                    'days_held': (date - entry_date).days,
                    'exit_reason': reason
                })
                cash = proceeds
                shares = 0.0

        equity_curve.append(cash + shares * close)

    if shares > 0:
        last_close = float(df['Close'].iloc[-1])
        cash = shares * last_close
        ret_pct = (last_close / entry_price - 1.0) * 100.0
        trades.append({'return_pct': ret_pct, 'days_held': 0, 'exit_reason': 'End'})

    final_equity = cash
    trades_df = pd.DataFrame(trades)

    # Buy & Hold
    bh_final = initial_capital * (df['Close'].iloc[-1] / df['Close'].iloc[start_i])

    years = (df.index[-1] - df.index[start_i]).days / 365.25
    cagr = (final_equity / initial_capital) ** (1/years) - 1 if years > 0 else 0
    bh_cagr = (bh_final / initial_capital) ** (1/years) - 1 if years > 0 else 0

    eq = pd.Series(equity_curve)
    max_dd = ((eq / eq.cummax()) - 1).min() if len(eq) > 0 else 0

    win_rate = (trades_df['return_pct'] > 0).mean() if len(trades_df) > 0 else 0
    avg_trade = trades_df['return_pct'].mean() if len(trades_df) > 0 else 0

    return {
        'ticker': None,
        'final_equity': round(final_equity, 0),
        'cagr': cagr,
        'max_dd': max_dd,
        'bh_final': round(bh_final, 0),
        'bh_cagr': bh_cagr,
        'num_trades': len(trades_df),
        'win_rate': win_rate,
        'avg_trade': avg_trade,
        'beats_bh': final_equity > bh_final
    }


# ====================== RUN TEST ======================
print("Running Multi-Stock Tester...")
print(f"Strategy: Z < {Z_ENTRY} | {ATR_MULT}×ATR Trail (Close) | Z > {Z_EXIT} exit")
print(f"Capital: ${INITIAL_CAPITAL} | From {START_DATE}\n")

results = []

for i, ticker in enumerate(TICKERS, 1):
    print(f"[{i:2}/{len(TICKERS)}] {ticker:<6} ", end="")
    df = get_data(ticker, START_DATE)
    
    if df is None:
        print("→ SKIPPED")
        continue
    
    res = run_strategy(df)
    res['ticker'] = ticker
    results.append(res)
    
    beat = "YES" if res['beats_bh'] else "no"
    print(f"→ ${res['final_equity']:>6,.0f}  | CAGR {res['cagr']*100:5.1f}% | DD {res['max_dd']*100:5.1f}% | Trades {res['num_trades']:3} | Beats BH: {beat}")

# Results table
summary = pd.DataFrame(results)
summary = summary.sort_values('final_equity', ascending=False).reset_index(drop=True)

print("\n" + "="*95)
print("FINAL RANKING (by Final Equity)")
print("="*95)
print(f"{'Rank':<5} {'Ticker':<7} {'Final $':>9} {'CAGR':>8} {'MaxDD':>8} {'BH $':>9} {'Trades':>7} {'Win%':>7} {'Avg%':>7} {'Beats?':>8}")
print("-"*95)

for idx, row in summary.iterrows():
    print(f"{idx+1:<5} {row['ticker']:<7} ${row['final_equity']:>7,.0f} {row['cagr']*100:>7.1f}% "
          f"{row['max_dd']*100:>7.1f}% ${row['bh_final']:>7,.0f} {row['num_trades']:>7} "
          f"{row['win_rate']*100:>6.1f}% {row['avg_trade']:>6.2f}% "
          f"{'YES' if row['beats_bh'] else 'no':>8}")

print("="*95)
print(f"\nStocks that beat Buy & Hold : {summary['beats_bh'].sum()} / {len(summary)}")
print(f"Average Strategy CAGR      : {summary['cagr'].mean()*100:.1f}%")
print(f"Average Max Drawdown       : {summary['max_dd'].mean()*100:.1f}%")

print("\nTOP 5 performers:")
print(summary[['ticker', 'final_equity', 'cagr', 'max_dd', 'win_rate', 'beats_bh']].head(5).to_string(index=False))

Running Multi-Stock Tester...
Strategy: Z < -1.5 | 2.0×ATR Trail (Close) | Z > 0.0 exit
Capital: $1000.0 | From 2025-01-01

[ 1/17] NOW    → $   831  | CAGR -12.2% | DD -39.3% | Trades  19 | Beats BH: YES
[ 2/17] CRM    → $   834  | CAGR -12.0% | DD -29.8% | Trades  20 | Beats BH: YES
[ 3/17] ADBE   → $   656  | CAGR -25.6% | DD -43.5% | Trades  18 | Beats BH: YES
[ 4/17] SNOW   → $   835  | CAGR -11.9% | DD -36.1% | Trades  12 | Beats BH: no
[ 5/17] DDOG   → $ 1,011  | CAGR   0.8% | DD -24.2% | Trades  12 | Beats BH: no
[ 6/17] CRWD   → $ 1,059  | CAGR   4.1% | DD -35.5% | Trades  12 | Beats BH: no
[ 7/17] NET    → $ 1,303  | CAGR  20.4% | DD -21.6% | Trades  10 | Beats BH: no
[ 8/17] PANW   → $ 1,128  | CAGR   8.8% | DD -22.6% | Trades  13 | Beats BH: no
[ 9/17] GOOGL  → $ 1,058  | CAGR   4.0% | DD -13.5% | Trades  15 | Beats BH: no
[10/17] META   → $ 1,424  | CAGR  28.1% | DD -15.4% | Trades  16 | Beats BH: YES
[11/17] AMZN   → $   937  | CAGR  -4.5% | DD -18.3% | Trades  15 | Beats

In [3]:
["NOW", "NVDA", "META", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "NET", "CRM"]

# ============================================================
# REFINED MULTI-STOCK TESTER
# Quality + Analyst Bullish Universe
# Original vs Trend Filter
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
START_DATE      = "2025-01-01"          # Change to "2024-01-01" if you want more data
INITIAL_CAPITAL = 1000.0

Z_ENTRY         = -1.5
Z_EXIT          = 0.0
ATR_MULT        = 2.0
SMA_WINDOW      = 20
ATR_WINDOW      = 14
TREND_SMA       = 100                   # Trend filter

# High Quality + Analyst Bullish list
TICKERS = ["NOW", "NVDA", "META", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "NET", "CRM"]
# ======================================================


def get_data(ticker, start):
    try:
        df = yf.download(ticker, start=start, multi_level_index=False,
                         progress=False, auto_adjust=True)
        if df.empty or len(df) < 60:
            return None
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
        df.index = pd.to_datetime(df.index)
        return df
    except:
        return None


def add_indicators(df):
    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']
    df['trend_sma'] = df['Close'].rolling(TREND_SMA).mean()

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()
    return df


def run_strategy(df, use_trend_filter=False, initial_capital=INITIAL_CAPITAL):
    df = add_indicators(df)

    cash = float(initial_capital)
    shares = 0.0
    entry_price = 0.0
    stop_price = 0.0
    entry_date = None
    trades = []
    equity_curve = []

    start_i = max(SMA_WINDOW, ATR_WINDOW, TREND_SMA if use_trend_filter else 0) + 1

    for i in range(start_i, len(df)):
        date = df.index[i]
        close = float(df['Close'].iloc[i])
        z = float(df['zscore'].iloc[i]) if pd.notna(df['zscore'].iloc[i]) else 0.0
        atr = float(df['atr'].iloc[i]) if pd.notna(df['atr'].iloc[i]) else 0.0
        trend_ok = True
        if use_trend_filter:
            trend_sma = float(df['trend_sma'].iloc[i]) if pd.notna(df['trend_sma'].iloc[i]) else 0
            trend_ok = close > trend_sma

        # Entry
        if shares == 0 and z < Z_ENTRY and atr > 0 and cash > 10 and trend_ok:
            shares = cash / close
            entry_price = close
            stop_price = close - (ATR_MULT * atr)
            entry_date = date
            cash = 0.0

        # Manage
        elif shares > 0:
            new_stop = close - (ATR_MULT * atr)
            if new_stop > stop_price:
                stop_price = new_stop

            reason = None
            if close <= stop_price:
                reason = "ATR Trail"
            elif z >= Z_EXIT:
                reason = "Z > 0"

            if reason:
                proceeds = shares * close
                ret_pct = (close / entry_price - 1) * 100
                trades.append({'return_pct': ret_pct, 'days_held': (date - entry_date).days})
                cash = proceeds
                shares = 0.0

        equity_curve.append(cash + shares * close)

    if shares > 0:
        last_close = float(df['Close'].iloc[-1])
        cash = shares * last_close
        ret_pct = (last_close / entry_price - 1) * 100
        trades.append({'return_pct': ret_pct, 'days_held': 0})

    final_equity = cash
    trades_df = pd.DataFrame(trades)

    bh_final = initial_capital * (df['Close'].iloc[-1] / df['Close'].iloc[start_i])
    years = max((df.index[-1] - df.index[start_i]).days / 365.25, 0.1)
    cagr = (final_equity / initial_capital) ** (1/years) - 1
    max_dd = ((pd.Series(equity_curve) / pd.Series(equity_curve).cummax()) - 1).min() if equity_curve else 0
    win_rate = (trades_df['return_pct'] > 0).mean() if len(trades_df) > 0 else 0
    num_trades = len(trades_df)

    return {
        'final_equity': round(final_equity),
        'cagr': cagr,
        'max_dd': max_dd,
        'bh_final': round(bh_final),
        'num_trades': num_trades,
        'win_rate': win_rate,
        'beats_bh': final_equity > bh_final
    }


# ====================== EXECUTE ======================
print("="*90)
print(f"REFINED QUALITY + ANALYST BULLISH TESTER")
print(f"Start Date: {START_DATE}  |  Trend Filter: Close > {TREND_SMA} SMA")
print("="*90)

results = []

for ticker in TICKERS:
    print(f"\n{ticker}:")
    df = get_data(ticker, START_DATE)
    if df is None:
        print("  No data")
        continue

    res_o = run_strategy(df, use_trend_filter=False)
    res_o['ticker'] = ticker
    res_o['version'] = "Original"

    res_t = run_strategy(df, use_trend_filter=True)
    res_t['ticker'] = ticker
    res_t['version'] = "Trend Filter"

    results.extend([res_o, res_t])

    print(f"  Original     → ${res_o['final_equity']:>5} | CAGR {res_o['cagr']*100:5.1f}% | DD {res_o['max_dd']*100:5.1f}% | Trades {res_o['num_trades']:2} | Win {res_o['win_rate']*100:4.0f}% | Beats BH: {res_o['beats_bh']}")
    print(f"  Trend Filter → ${res_t['final_equity']:>5} | CAGR {res_t['cagr']*100:5.1f}% | DD {res_t['max_dd']*100:5.1f}% | Trades {res_t['num_trades']:2} | Win {res_t['win_rate']*100:4.0f}% | Beats BH: {res_t['beats_bh']}")

# Summary
df_res = pd.DataFrame(results)

print("\n\n" + "="*90)
print("TREND FILTER VERSION - RANKED BY FINAL EQUITY")
print("="*90)
filt = df_res[df_res['version'] == "Trend Filter"].sort_values('final_equity', ascending=False)
print(filt[['ticker', 'final_equity', 'cagr', 'max_dd', 'num_trades', 'win_rate', 'beats_bh']].to_string(index=False))

print("\n\nORIGINAL VERSION - RANKED BY FINAL EQUITY")
print("="*90)
orig = df_res[df_res['version'] == "Original"].sort_values('final_equity', ascending=False)
print(orig[['ticker', 'final_equity', 'cagr', 'max_dd', 'num_trades', 'win_rate', 'beats_bh']].to_string(index=False))

REFINED QUALITY + ANALYST BULLISH TESTER
Start Date: 2025-01-01  |  Trend Filter: Close > 100 SMA

NOW:
  Original     → $  831 | CAGR -12.2% | DD -39.3% | Trades 19 | Win   53% | Beats BH: True
  Trend Filter → $ 1036 | CAGR   3.3% | DD  -3.6% | Trades  2 | Win  100% | Beats BH: True

NVDA:
  Original     → $ 1435 | CAGR  28.8% | DD  -9.4% | Trades 11 | Win   82% | Beats BH: False
  Trend Filter → $  982 | CAGR  -1.6% | DD  -9.4% | Trades  3 | Win   67% | Beats BH: False

META:
  Original     → $ 1424 | CAGR  28.1% | DD -15.4% | Trades 16 | Win   62% | Beats BH: True
  Trend Filter → $ 1142 | CAGR  12.7% | DD  -4.0% | Trades  3 | Win   67% | Beats BH: True

MSFT:
  Original     → $ 1096 | CAGR   6.6% | DD -12.8% | Trades 17 | Win   59% | Beats BH: True
  Trend Filter → $ 1010 | CAGR   0.9% | DD  -2.9% | Trades  1 | Win  100% | Beats BH: True

GOOGL:
  Original     → $ 1058 | CAGR   4.0% | DD -13.5% | Trades 15 | Win   53% | Beats BH: False
  Trend Filter → $ 1015 | CAGR   1.4% | DD -1

In [4]:
# ============================================================
# REFINED TESTER - Mild Trend Filter (50-SMA)
# Complete results for all stocks
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
START_DATE      = "2025-01-01"          # ← change to "2024-01-01" if you want more history
INITIAL_CAPITAL = 1000.0

Z_ENTRY         = -1.5
Z_EXIT          = 0.0
ATR_MULT        = 2.0
SMA_WINDOW      = 20
ATR_WINDOW      = 14
TREND_SMA       = 50                    # Milder filter

TICKERS = ["NOW", "NVDA", "META", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "NET", "CRM"]
# ======================================================


def get_data(ticker, start):
    try:
        df = yf.download(ticker, start=start, multi_level_index=False,
                         progress=False, auto_adjust=True)
        if df.empty or len(df) < 80:
            return None
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        print(f"Error {ticker}: {e}")
        return None


def add_indicators(df):
    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']
    df['trend_sma'] = df['Close'].rolling(TREND_SMA).mean()

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()
    return df


def run_strategy(df, use_trend_filter=False, initial_capital=INITIAL_CAPITAL):
    df = add_indicators(df)

    cash = float(initial_capital)
    shares = 0.0
    entry_price = 0.0
    stop_price = 0.0
    entry_date = None
    trades = []
    equity_curve = []

    min_bars = max(SMA_WINDOW, ATR_WINDOW, TREND_SMA if use_trend_filter else 0) + 1

    for i in range(min_bars, len(df)):
        date = df.index[i]
        close = float(df['Close'].iloc[i])
        z = float(df['zscore'].iloc[i]) if pd.notna(df['zscore'].iloc[i]) else 0.0
        atr = float(df['atr'].iloc[i]) if pd.notna(df['atr'].iloc[i]) else 0.0

        trend_ok = True
        if use_trend_filter:
            trend_val = df['trend_sma'].iloc[i]
            trend_ok = pd.notna(trend_val) and close > float(trend_val)

        # Entry
        if shares == 0 and z < Z_ENTRY and atr > 0 and cash > 10 and trend_ok:
            shares = cash / close
            entry_price = close
            stop_price = close - (ATR_MULT * atr)
            entry_date = date
            cash = 0.0

        # Manage
        elif shares > 0:
            new_stop = close - (ATR_MULT * atr)
            if new_stop > stop_price:
                stop_price = new_stop

            reason = None
            if close <= stop_price:
                reason = "ATR Trail"
            elif z >= Z_EXIT:
                reason = "Z > 0"

            if reason:
                proceeds = shares * close
                ret_pct = (close / entry_price - 1.0) * 100.0
                trades.append({
                    'return_pct': ret_pct,
                    'days_held': (date - entry_date).days
                })
                cash = proceeds
                shares = 0.0

        equity_curve.append(cash + shares * close)

    if shares > 0:
        last_close = float(df['Close'].iloc[-1])
        cash = shares * last_close
        ret_pct = (last_close / entry_price - 1.0) * 100.0
        trades.append({'return_pct': ret_pct, 'days_held': 0})

    final_equity = cash
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame(columns=['return_pct'])

    start_price = float(df['Close'].iloc[min_bars])
    end_price = float(df['Close'].iloc[-1])
    bh_final = initial_capital * (end_price / start_price)

    years = max((df.index[-1] - df.index[min_bars]).days / 365.25, 0.1)
    cagr = (final_equity / initial_capital) ** (1 / years) - 1

    max_dd = 0.0
    if equity_curve:
        eq = pd.Series(equity_curve)
        max_dd = ((eq / eq.cummax()) - 1).min()

    win_rate = (trades_df['return_pct'] > 0).mean() if len(trades_df) > 0 else 0.0
    avg_trade = trades_df['return_pct'].mean() if len(trades_df) > 0 else 0.0
    num_trades = len(trades_df)

    return {
        'final_equity': round(final_equity),
        'cagr': cagr,
        'max_dd': max_dd,
        'bh_final': round(bh_final),
        'num_trades': num_trades,
        'win_rate': win_rate,
        'avg_trade': avg_trade,
        'beats_bh': final_equity > bh_final
    }


# ====================== RUN ======================
print("=" * 100)
print(f"MILD TREND FILTER TEST  |  Start: {START_DATE}  |  Filter: Close > {TREND_SMA}-SMA")
print(f"Tickers: {', '.join(TICKERS)}")
print("=" * 100)

all_rows = []

for ticker in TICKERS:
    print(f"\n{ticker}")
    print("-" * 50)
    df = get_data(ticker, START_DATE)

    if df is None:
        print("  → Skipped (insufficient data)")
        continue

    res_orig = run_strategy(df, use_trend_filter=False)
    res_orig['ticker'] = ticker
    res_orig['version'] = "Original"
    all_rows.append(res_orig)

    res_filt = run_strategy(df, use_trend_filter=True)
    res_filt['ticker'] = ticker
    res_filt['version'] = "Trend50"
    all_rows.append(res_filt)

    print(f"  Original  → ${res_orig['final_equity']:>6} | CAGR {res_orig['cagr']*100:6.1f}% | "
          f"DD {res_orig['max_dd']*100:6.1f}% | Trades {res_orig['num_trades']:2} | "
          f"Win {res_orig['win_rate']*100:5.1f}% | BeatsBH: {res_orig['beats_bh']}")

    print(f"  Trend50   → ${res_filt['final_equity']:>6} | CAGR {res_filt['cagr']*100:6.1f}% | "
          f"DD {res_filt['max_dd']*100:6.1f}% | Trades {res_filt['num_trades']:2} | "
          f"Win {res_filt['win_rate']*100:5.1f}% | BeatsBH: {res_filt['beats_bh']}")


# ====================== FULL TABLES ======================
df_res = pd.DataFrame(all_rows)

print("\n\n" + "=" * 100)
print("FULL TABLE — ORIGINAL (No Filter)")
print("=" * 100)
orig = df_res[df_res['version'] == "Original"].sort_values('final_equity', ascending=False)
print(f"{'Ticker':<8} {'Final$':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>7} {'Win%':>7} {'Avg%':>8} {'BeatsBH':>9}")
print("-" * 75)
for _, r in orig.iterrows():
    print(f"{r['ticker']:<8} ${r['final_equity']:>6} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% "
          f"{r['num_trades']:>7} {r['win_rate']*100:>6.1f}% {r['avg_trade']:>7.2f}% {str(r['beats_bh']):>9}")

print("\n\n" + "=" * 100)
print("FULL TABLE — MILD TREND FILTER (Close > 50-SMA)")
print("=" * 100)
filt = df_res[df_res['version'] == "Trend50"].sort_values('final_equity', ascending=False)
print(f"{'Ticker':<8} {'Final$':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>7} {'Win%':>7} {'Avg%':>8} {'BeatsBH':>9}")
print("-" * 75)
for _, r in filt.iterrows():
    print(f"{r['ticker']:<8} ${r['final_equity']:>6} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% "
          f"{r['num_trades']:>7} {r['win_rate']*100:>6.1f}% {r['avg_trade']:>7.2f}% {str(r['beats_bh']):>9}")

print("\n\n" + "=" * 100)
print("SIDE-BY-SIDE (Final Equity)")
print("=" * 100)
comp = df_res.pivot(index='ticker', columns='version', values='final_equity')
comp['Diff'] = comp['Trend50'] - comp['Original']
comp = comp.sort_values('Original', ascending=False)
print(comp.round(0).to_string())

print("\nDone.")

MILD TREND FILTER TEST  |  Start: 2025-01-01  |  Filter: Close > 50-SMA
Tickers: NOW, NVDA, META, MSFT, GOOGL, PANW, CRWD, DDOG, NET, CRM

NOW
--------------------------------------------------
  Original  → $   831 | CAGR  -12.2% | DD  -39.3% | Trades 19 | Win  52.6% | BeatsBH: True
  Trend50   → $  1007 | CAGR    0.5% | DD   -3.6% | Trades  1 | Win 100.0% | BeatsBH: True

NVDA
--------------------------------------------------
  Original  → $  1435 | CAGR   28.8% | DD   -9.4% | Trades 11 | Win  81.8% | BeatsBH: False
  Trend50   → $   956 | CAGR   -3.4% | DD   -9.4% | Trades  2 | Win  50.0% | BeatsBH: False

META
--------------------------------------------------
  Original  → $  1424 | CAGR   28.1% | DD  -15.4% | Trades 16 | Win  62.5% | BeatsBH: True
  Trend50   → $  1098 | CAGR    7.4% | DD   -4.1% | Trades  3 | Win  66.7% | BeatsBH: False

MSFT
--------------------------------------------------
  Original  → $  1096 | CAGR    6.6% | DD  -12.8% | Trades 17 | Win  58.8% | BeatsBH: 